# SIM Intervention Demand Forecasting

**YCIT 440 – AI and Data Science Capstone Project**

**Team 4**

## 3. Feature Engineering


### 3.1. Project Objective

The objective of this stage is to create predictive features for forecasting the total number of recorded daily interventions for each of SIM operational Divisions 1 through 6.

The input is `SIM_Daily_Interventions.csv`, the daily division-level dataset prepared during Stage 1.

Predictive features will include calendar information, historical intervention counts, and rolling statistics.

For each forecast day D, the forecast is prepared on D−1 using historical intervention counts available through D−3, consistent with the revised Business Problem Framing Report.

Calendar information for the forecast day can be used because it is known in advance.

The resulting features will support the development and evaluation of Linear Regression, LightGBM, and CatBoost forecasting models.

Feature engineering will not use intervention counts from D, D−1, or D−2 as predictors.

This stage will produce two datasets:

- `SIM_Features_Complete.csv`: Preserves all division-day observations, including rows with missing target values.
- `SIM_Features_Modeling.csv`: Contains observations with known target values for supervised modeling.

The predictive usefulness of these features will be evaluated in subsequent stages.


### 3.2. Import Required Libraries

We will use pandas to organize the daily intervention data and construct time-based features. NumPy will support numerical operations and missing-value handling.

Additional libraries will be introduced only when needed.

In [ ]:

import pandas as pd
import numpy as np


### 3.3. Load the Prepared Daily Dataset

The daily intervention dataset was created and structurally validated during Stage 1: Modeling Dataset — Data Preparation.

It contains one row per operational division and calendar day, including the six missing target values for December 31, 2024.

We will load the dataset and verify that its structure is consistent with the previous stage before creating predictive features.

In [ ]:

# Load the prepared daily intervention dataset
input_file = "../../data/interim/modelling/SIM_Daily_Interventions.csv"

df_daily = pd.read_csv(input_file, parse_dates=["Date"])

# Verify the dataset
print("Dataset dimensions:", df_daily.shape)
print("First date:", df_daily["Date"].min())
print("Last date:", df_daily["Date"].max())
print("Missing target values:", df_daily["Intervention_Count"].isna().sum())

# Display the first six observations
display(df_daily.head(6))


Dataset dimensions: (14754, 3)
First date: 2020-01-01 00:00:00
Last date: 2026-09-24 00:00:00
Missing target values: 6


,Date,DIVISION,Intervention_Count
0,2020-01-01,1,35.0
1,2020-01-01,2,67.0
2,2020-01-01,3,55.0
3,2020-01-01,4,70.0
4,2020-01-01,5,64.0
5,2020-01-01,6,69.0


### 3.4. Create Calendar Features

Calendar features provide information about the date being forecast and may help the models identify recurring temporal patterns in intervention demand.

We will create the following features:

- `DayOfWeek`: Day of the week, from Monday (0) to Sunday (6).
- `Month`: Calendar month, from 1 to 12.
- `DayOfMonth`: Day of the month, from 1 to 31.
- `IsWeekend`: Indicates whether the forecast date falls on Saturday or Sunday.
- `DayOfYear`: Position of the date within the calendar year.

These features are known in advance and do not depend on actual intervention counts from the forecast date. Therefore, they do not introduce target leakage.


In [ ]:

# Create a separate DataFrame for feature engineering
df_features = df_daily.copy()

# Create calendar features
df_features["DayOfWeek"] = df_features["Date"].dt.dayofweek

df_features["Month"] = df_features["Date"].dt.month

df_features["DayOfMonth"] = df_features["Date"].dt.day

df_features["IsWeekend"] = (
    df_features["DayOfWeek"] >= 5
).astype(int)

df_features["DayOfYear"] = df_features["Date"].dt.dayofyear

# Display the updated dataset
print("Dataset dimensions:", df_features.shape)

display(df_features.head(12))


Dataset dimensions: (14754, 8)


,Date,DIVISION,Intervention_Count,DayOfWeek,Month,DayOfMonth,IsWeekend,DayOfYear
0,2020-01-01,1,35.0,2,1,1,0,1
1,2020-01-01,2,67.0,2,1,1,0,1
2,2020-01-01,3,55.0,2,1,1,0,1
3,2020-01-01,4,70.0,2,1,1,0,1
4,2020-01-01,5,64.0,2,1,1,0,1
5,2020-01-01,6,69.0,2,1,1,0,1
6,2020-01-02,1,24.0,3,1,2,0,2
7,2020-01-02,2,66.0,3,1,2,0,2
8,2020-01-02,3,58.0,3,1,2,0,2
9,2020-01-02,4,54.0,3,1,2,0,2


**Methodological Observation**

Five calendar features were created: `DayOfWeek`, `Month`, `DayOfMonth`, `IsWeekend`, and `DayOfYear`.

These are candidate predictors that may help models capture weekly, monthly, or seasonal differences in daily intervention demand. Their usefulness has not yet been established and will be assessed during model validation and comparison.

All five describe the forecast date and are known when the forecast is prepared on D−1. They therefore do not require intervention counts beyond the D−3 information cutoff.

The features are currently represented as integers. This encoding may have different implications for linear and tree-based models. Categorical or cyclic encoding choices should be evaluated during model training. `IsWeekend` is derived from `DayOfWeek`, and `DayOfYear` may vary for the same calendar date in leap years.

The resulting dataset contains 14,754 observations and eight columns, preserving the original division-day structure.


### 3.5. Verify Calendar Features and Target Integrity

We will verify that the calendar features contain valid values and that the original intervention counts remain unchanged.

The verification will check the ranges of the calendar variables, the number of missing target values, and the total number of recorded interventions.

This will confirm that adding calendar features has preserved the original modeling dataset.

In [ ]:

# Verify calendar feature ranges
print("DayOfWeek range:", df_features["DayOfWeek"].min(), "to", df_features["DayOfWeek"].max())
print("Month range:", df_features["Month"].min(), "to", df_features["Month"].max())
print("DayOfMonth range:", df_features["DayOfMonth"].min(), "to", df_features["DayOfMonth"].max())
print("IsWeekend values:", sorted(df_features["IsWeekend"].unique()))
print("DayOfYear range:", df_features["DayOfYear"].min(), "to", df_features["DayOfYear"].max())

# Verify that target values are preserved
print("\nMissing target values:", df_features["Intervention_Count"].isna().sum())
print("Total recorded interventions:", int(df_features["Intervention_Count"].sum()))

# Verify dataset dimensions
print("Dataset dimensions:", df_features.shape)


# Verify that all original columns remain unchanged
original_columns = ["Date", "DIVISION", "Intervention_Count"]

pd.testing.assert_frame_equal(
    df_features[original_columns],
    df_daily[original_columns]
)

print("Original dataset integrity: PASSED")



DayOfWeek range: 0 to 6
Month range: 1 to 12
DayOfMonth range: 1 to 31
IsWeekend values: [np.int64(0), np.int64(1)]
DayOfYear range: 1 to 366

Missing target values: 6
Total recorded interventions: 753813
Dataset dimensions: (14754, 8)
Original dataset integrity: PASSED



**Methodological Observation**

The calendar features have the expected value ranges, and the dataset retains 14,754 observations, eight columns, and six missing target values.

The total recorded intervention count is 753,813, consistent with the prepared daily dataset.

A direct comparison of the original columns confirms that dates, divisions, intervention counts, and missing target positions remain unchanged after creating the calendar features.

Range checks verify valid feature values but do not establish that every calendar value is correctly assigned or that the features improve forecasting performance. Their predictive usefulness will be evaluated in subsequent stages.


### 3.6. Create Lag Features

Lag features represent historical intervention counts from previous calendar days for the same operational division.

We will create four candidate lag features: `Lag_3`, `Lag_7`, `Lag_14`, and `Lag_28`.

All selected lags respect the D−3 information cutoff established in the revised Business Problem Framing Report.

The lag features will be calculated separately for each division to prevent intervention counts from different divisions from being mixed.

Missing historical counts will remain missing rather than being replaced with zero.

In [ ]:

# Sort records by division and date
df_features = df_features.sort_values(
    ["DIVISION", "Date"]
).reset_index(drop=True)


# Confirm that each division has consecutive calendar days
day_steps = df_features.groupby("DIVISION")["Date"].diff().dropna()

assert day_steps.eq(pd.Timedelta(days=1)).all(), (
    "Missing or repeated calendar dates within a division."
)

print("Daily continuity within divisions: PASSED")


# Define candidate lag values
lag_days = [3, 7, 14, 28]

# Create lag features for each division
for lag in lag_days:
    df_features[f"Lag_{lag}"] = (
        df_features.groupby("DIVISION")["Intervention_Count"]
        .shift(lag)
    )

# Display dataset dimensions
print("Dataset dimensions:", df_features.shape)

# Display example observations
display(
    df_features[
        df_features["DIVISION"] == 1
    ].head(35)
)


Daily continuity within divisions: PASSED
Dataset dimensions: (14754, 12)


,Date,DIVISION,Intervention_Count,DayOfWeek,Month,DayOfMonth,IsWeekend,DayOfYear,Lag_3,Lag_7,Lag_14,Lag_28
0,2020-01-01,1,35.0,2,1,1,0,1,NaN,NaN,NaN,NaN
1,2020-01-02,1,24.0,3,1,2,0,2,NaN,NaN,NaN,NaN
2,2020-01-03,1,18.0,4,1,3,0,3,NaN,NaN,NaN,NaN
3,2020-01-04,1,22.0,5,1,4,1,4,35.0,NaN,NaN,NaN
4,2020-01-05,1,24.0,6,1,5,1,5,24.0,NaN,NaN,NaN
5,2020-01-06,1,27.0,0,1,6,0,6,18.0,NaN,NaN,NaN
6,2020-01-07,1,33.0,1,1,7,0,7,22.0,NaN,NaN,NaN
7,2020-01-08,1,34.0,2,1,8,0,8,24.0,35.0,NaN,NaN
8,2020-01-09,1,24.0,3,1,9,0,9,27.0,24.0,NaN,NaN
9,2020-01-10,1,34.0,4,1,10,0,10,33.0,18.0,NaN,NaN


### 3.7. Verify Lag Features and Information Cutoff

We will verify that the lag features correspond to the correct historical intervention counts for a selected forecast date and division.

The verification will use January 19, 2024, for Division 1.

Each lag value will be compared with its corresponding historical observation to confirm that the features were calculated correctly and respect the D−3 information cutoff.

We will also examine the lag values around the known missing date of December 31, 2024, to verify that calendar-based shifts preserve missing observations rather than substituting values from other dates.

In [ ]:

# Define the example forecast
forecast_date = pd.Timestamp("2024-01-19")
division = 1

# Select the forecast observation
forecast_row = df_features[
    (df_features["Date"] == forecast_date) &
    (df_features["DIVISION"] == division)
]

print("Forecast date:", forecast_date.date())
print("Division:", division)

# Verify each lag against its historical count
for lag in lag_days:

    historical_date = forecast_date - pd.Timedelta(days=lag)

    historical_row = df_daily[
        (df_daily["Date"] == historical_date) &
        (df_daily["DIVISION"] == division)
    ]

    actual_historical_count = historical_row["Intervention_Count"].iloc[0]
    lag_value = forecast_row[f"Lag_{lag}"].iloc[0]

    assert np.isclose(actual_historical_count, lag_value), (
        f"Lag_{lag} does not match the historical count for "
        f"Division {division} on {forecast_date.date()}."
    )

    print(
        f"Lag_{lag}:",
        "Historical date:", historical_date.date(),
        "| Historical count:", actual_historical_count,
        "| Feature value:", lag_value,
        "| Match:", np.isclose(actual_historical_count, lag_value)
    )


Forecast date: 2024-01-19
Division: 1
Lag_3: Historical date: 2024-01-16 | Historical count: 31.0 | Feature value: 31.0 | Match: True
Lag_7: Historical date: 2024-01-12 | Historical count: 30.0 | Feature value: 30.0 | Match: True
Lag_14: Historical date: 2024-01-05 | Historical count: 30.0 | Feature value: 30.0 | Match: True
Lag_28: Historical date: 2023-12-22 | Historical count: 33.0 | Feature value: 33.0 | Match: True


In [ ]:

# Check how the missing date affects a lag feature
missing_date = pd.Timestamp("2024-12-31")
example_forecast_date = missing_date + pd.Timedelta(days=3)

# Select Division 1 on January 3, 2025
example_row = df_features[
    (df_features["Date"] == example_forecast_date) &
    (df_features["DIVISION"] == 1)
]

print("Forecast date:", example_forecast_date.date())
print("Lag_3 historical date:", missing_date.date())
print("Lag_3 value:", example_row["Lag_3"].iloc[0])
print("Is Lag_3 missing?", pd.isna(example_row["Lag_3"].iloc[0]))

# Verify missing Lag_3 values across all six divisions
lag_3_values = df_features.loc[
    df_features["Date"] == example_forecast_date, "Lag_3"
]

assert len(lag_3_values) == 6 and lag_3_values.isna().all(), (
    "Lag_3 should be missing for all six divisions on 2025-01-03."
)

print("Missing-date lag propagation across divisions: PASSED")



Forecast date: 2025-01-03
Lag_3 historical date: 2024-12-31
Lag_3 value: nan
Is Lag_3 missing? True
Missing-date lag propagation across divisions: PASSED


**Methodological Observation**

The four lag features (`Lag_3`, `Lag_7`, `Lag_14`, and `Lag_28`) were verified against their corresponding historical intervention counts for Division 1 on January 19, 2024.

All four values matched the original records and respected the D−3 information cutoff.

An additional check confirmed that `Lag_3` for January 3, 2025, remains missing because its historical reference date, December 31, 2024, has no recorded intervention count.

These results confirm that the tested lag features preserve calendar alignment and missing historical values without introducing future intervention counts.

### 3.8. Create Rolling Average Features

Rolling average features summarize recent historical intervention counts for each operational division.

We will create two candidate features:

- `Rolling_Mean_7`: Average intervention count over the seven calendar days ending at D−3.
- `Rolling_Mean_14`: Average intervention count over the fourteen calendar days ending at D−3.

The calculations will be performed separately for each division. Historical counts from D−2, D−1, and the forecast day D will not be included.

A rolling average will remain missing if any intervention count within its required historical window is unavailable. This preserves the distinction between complete historical information and missing data.

In [ ]:

# Define rolling window sizes
rolling_windows = [7, 14]

# Create rolling averages for each division
for window in rolling_windows:

    df_features[f"Rolling_Mean_{window}"] = (
        df_features.groupby("DIVISION")["Intervention_Count"]
        .transform(
            lambda counts: counts.shift(3).rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

# Display dataset dimensions
print("Dataset dimensions:", df_features.shape)

# Display example observations for Division 1
display(
    df_features[
        df_features["DIVISION"] == 1
    ][
        ["Date", "Intervention_Count", "Rolling_Mean_7", "Rolling_Mean_14"]
    ].head(20)
)


Dataset dimensions: (14754, 14)


,Date,Intervention_Count,Rolling_Mean_7,Rolling_Mean_14
0,2020-01-01,35.0,NaN,NaN
1,2020-01-02,24.0,NaN,NaN
2,2020-01-03,18.0,NaN,NaN
3,2020-01-04,22.0,NaN,NaN
4,2020-01-05,24.0,NaN,NaN
5,2020-01-06,27.0,NaN,NaN
6,2020-01-07,33.0,NaN,NaN
7,2020-01-08,34.0,NaN,NaN
8,2020-01-09,24.0,NaN,NaN
9,2020-01-10,34.0,26.142857,NaN


### 3.9. Verify Rolling Average Calculations

We will verify the rolling average features using Division 1 and January 19, 2024, as an example forecast.

For each rolling window, we will identify the historical intervention counts available through D−3 and manually calculate their average.

The manually calculated averages will be compared with the corresponding feature values to verify calendar alignment and confirm that no unavailable future counts were included.

In [ ]:

# Define the example forecast
forecast_date = pd.Timestamp("2024-01-19")
division = 1

# Calculate the latest allowed historical date
cutoff_date = forecast_date - pd.Timedelta(days=3)

# Select the forecast observation
forecast_row = df_features[
    (df_features["Date"] == forecast_date) &
    (df_features["DIVISION"] == division)
]

print("Forecast date:", forecast_date.date())
print("Division:", division)
print("Cutoff date:", cutoff_date.date())

# Verify each rolling average
for window in rolling_windows:

    # Calculate the beginning of the historical window
    start_date = cutoff_date - pd.Timedelta(days=window - 1)

    # Select historical intervention counts
    historical_counts = df_daily[
        (df_daily["DIVISION"] == division) &
        (df_daily["Date"] >= start_date) &
        (df_daily["Date"] <= cutoff_date)
    ]


    # Verify that the historical window is complete
    assert len(historical_counts) == window, (
        f"Rolling_Mean_{window}: expected {window} historical dates."
    )

    assert historical_counts["Intervention_Count"].notna().all(), (
        f"Rolling_Mean_{window}: incomplete historical counts."
    )


    # Calculate the average manually
    manual_average = historical_counts["Intervention_Count"].mean()

    # Retrieve the previously calculated feature
    feature_average = forecast_row[f"Rolling_Mean_{window}"].iloc[0]

    print(f"\nRolling_Mean_{window}")
    print("Historical period:", start_date.date(), "to", cutoff_date.date())
    print("Historical observations:", len(historical_counts))
    print("Manual average:", round(manual_average, 4))
    print("Feature average:", round(feature_average, 4))
    print("Match:", np.isclose(manual_average, feature_average))

    assert np.isclose(manual_average, feature_average), (
        f"Rolling_Mean_{window}: manual and feature averages do not match."
    )



Forecast date: 2024-01-19
Division: 1
Cutoff date: 2024-01-16

Rolling_Mean_7
Historical period: 2024-01-10 to 2024-01-16
Historical observations: 7
Manual average: 26.1429
Feature average: 26.1429
Match: True

Rolling_Mean_14
Historical period: 2024-01-03 to 2024-01-16
Historical observations: 14
Manual average: 27.0
Feature average: 27.0
Match: True


**Methodological Observation**

The rolling average features were verified using Division 1 and January 19, 2024, as an example forecast.

`Rolling_Mean_7` produced an average of 26.1429 interventions using seven historical observations from January 10 through January 16, 2024. `Rolling_Mean_14` produced an average of 27.0000 interventions using fourteen historical observations from January 3 through January 16, 2024.

Both manually calculated averages matched the corresponding feature values. The historical windows ended at D−3, confirming that the tested calculations did not use unavailable future intervention counts.

### 3.10. Verify Rolling Features Around Missing Historical Data

We will verify that rolling average features correctly preserve missing historical intervention counts.

December 31, 2024, is the only missing calendar day in the dataset. Any rolling window that includes this date must remain missing because the required historical information is incomplete.

This verification will confirm that the rolling calculations do not replace unknown intervention counts with zero or silently calculate averages from incomplete windows.

In [ ]:

# Select dates around the known missing historical day
start_date = "2025-01-03"
end_date = "2025-01-17"

# Select Division 1
missing_date_check = df_features[
    (df_features["DIVISION"] == 1) &
    (df_features["Date"] >= start_date) &
    (df_features["Date"] <= end_date)
]

# Display the rolling features
display(
    missing_date_check[
        ["Date", "Rolling_Mean_7", "Rolling_Mean_14"]
    ]
)


,Date,Rolling_Mean_7,Rolling_Mean_14
1829,2025-01-03,NaN,NaN
1830,2025-01-04,NaN,NaN
1831,2025-01-05,NaN,NaN
1832,2025-01-06,NaN,NaN
1833,2025-01-07,NaN,NaN
1834,2025-01-08,NaN,NaN
1835,2025-01-09,NaN,NaN
1836,2025-01-10,32.714286,NaN
1837,2025-01-11,34.571429,NaN
1838,2025-01-12,35.571429,NaN


**Methodological Observation**

The rolling average features correctly preserved missing historical information associated with December 31, 2024.

For Division 1, `Rolling_Mean_7` remained missing from January 3 through January 9, 2025, and became available again on January 10. `Rolling_Mean_14` remained missing through January 16 and became available on January 17.

These results confirm that rolling averages are calculated only when all required historical daily counts are available. Missing intervention counts are not replaced with zero or excluded to produce incomplete-window averages.

The calculations also preserve the D−3 historical information cutoff.

### 3.11. Review Feature Completeness

We will examine the number of missing values in each feature of the prepared dataset.

Missing values may result from insufficient historical observations at the beginning of the dataset or from the known missing intervention counts on December 31, 2024.

This analysis will help determine which observations have complete predictive information before model training.

No missing values will be replaced or removed at this stage.

In [ ]:

# Count missing values in each column
missing_values = df_features.isna().sum()

# Calculate missing percentages
missing_percentages = (
    missing_values / len(df_features) * 100
)

# Create a summary table
missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": missing_percentages
})

# Display the results
print("Missing values by column:")
display(missing_summary.round(2))

# Count observations with at least one missing feature
feature_columns = [
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "DayOfYear",
    "Lag_3",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14"
]

rows_with_missing_features = (
    df_features[feature_columns].isna().any(axis=1).sum()
)

print("Rows with at least one missing feature:", rows_with_missing_features)


Missing values by column:


,Missing_Count,Missing_Percentage
Date,0,0.00
DIVISION,0,0.00
Intervention_Count,6,0.04
DayOfWeek,0,0.00
Month,0,0.00
DayOfMonth,0,0.00
IsWeekend,0,0.00
DayOfYear,0,0.00
Lag_3,24,0.16
Lag_7,48,0.33


Rows with at least one missing feature: 258



**Methodological Observation**

The feature completeness analysis identified 258 observations with at least one missing predictive feature, representing approximately 1.75% of the 14,754 division-day observations.

All calendar features are complete. Missing historical features have two identified causes:

- Insufficient historical observations at the beginning of the dataset.
- Missing intervention counts for December 31, 2024.

The affected observations include 168 rows at the beginning of the dataset, 84 rows from January 3 through January 16, 2025, and six additional rows on January 28, 2025, where `Lag_28` references December 31, 2024.

The dataset also contains six observations with unknown target values.

Missing predictive features will be preserved during this stage. LightGBM and CatBoost can handle missing numerical values, while Linear Regression will require an appropriate missing-value strategy during model preparation.

Observations with unknown target values cannot be used for supervised model training or accuracy evaluation.

No predictive features will be imputed or removed during this section.


### 3.12. Prepare the Feature Dataset for Modeling

We will prepare a separate dataset containing observations with known intervention counts for supervised model development and evaluation.

Rows with missing predictive features will be retained at this stage. LightGBM and CatBoost can handle missing numerical features, while Linear Regression will require a separate missing-value strategy during model preparation.


The six observations with unknown target values will be excluded from the supervised modeling dataset, while the complete feature dataset will remain unchanged.

The chronological development, validation, and final evaluation periods will be handled in subsequent stages.

In [ ]:

# Keep the complete feature dataset unchanged
df_features_complete = df_features.copy()

# Create a dataset with known target values
df_modeling = df_features.dropna(
    subset=["Intervention_Count"]
).copy()

# Display dataset dimensions
print("Complete feature dataset:", df_features_complete.shape)
print("Supervised modeling dataset:", df_modeling.shape)

# Verify target values
print(
    "Missing targets in modeling dataset:",
    df_modeling["Intervention_Count"].isna().sum()
)

# Verify missing predictive features are preserved
print(
    "Rows with missing predictive features:",
    df_modeling[feature_columns].isna().any(axis=1).sum()
)


Complete feature dataset: (14754, 14)
Supervised modeling dataset: (14748, 14)
Missing targets in modeling dataset: 0
Rows with missing predictive features: 258


**Methodological Observation**

The supervised modeling dataset contains 14,748 observations with known intervention counts, while the complete feature dataset preserves all 14,754 division-day combinations.

Six observations with unknown target values were excluded from the supervised modeling dataset.

The 258 observations containing at least one missing predictive feature were retained for subsequent model-specific preparation. LightGBM and CatBoost can handle missing numerical features, while Linear Regression will require an appropriate preprocessing strategy.

Historical features were created before excluding observations with unknown targets, preserving the original calendar alignment.

This approach preserves available training information without treating unknown intervention counts as zero.

The complete feature dataset remains available for reference, while chronological data splitting and model training will be addressed in subsequent stages.

### 3.13. Verify Final Feature Structure

We will verify the final structure of the supervised modeling dataset before saving the feature engineering outputs.

The dataset should contain the forecasting target, the operational division identifier, calendar features, lag features, and rolling averages.

We will also confirm that each division-day combination appears only once and that the six operational divisions are represented.

This verification will help ensure that the feature dataset is structurally ready for chronological validation and model development.

In [ ]:

# Display all dataset columns
print("Modeling dataset columns:")
print(df_modeling.columns.tolist())

# Verify the number of predictive features
predictor_columns = ["DIVISION"] + feature_columns

print("\nNumber of predictive features:", len(predictor_columns))

# Check duplicate division-day combinations
duplicate_combinations = df_modeling.duplicated(
    subset=["Date", "DIVISION"]
).sum()

print("Duplicate date-division combinations:", duplicate_combinations)

# Verify operational divisions
print("Divisions:", sorted(df_modeling["DIVISION"].unique()))

# Display dataset dimensions
print("Modeling dataset dimensions:", df_modeling.shape)


# Validate the final predictor structure
assert len(predictor_columns) == 12 and set(predictor_columns).issubset(df_modeling.columns), (
    "Unexpected predictor structure."
)

# Validate unique division-day combinations
assert duplicate_combinations == 0, (
    "Duplicate division-day combinations found."
)

# Validate operational divisions
assert sorted(df_modeling["DIVISION"].unique().tolist()) == [1, 2, 3, 4, 5, 6], (
    "Unexpected operational divisions."
)

print("Final feature structure validation: PASSED")



Modeling dataset columns:
['Date', 'DIVISION', 'Intervention_Count', 'DayOfWeek', 'Month', 'DayOfMonth', 'IsWeekend', 'DayOfYear', 'Lag_3', 'Lag_7', 'Lag_14', 'Lag_28', 'Rolling_Mean_7', 'Rolling_Mean_14']

Number of predictive features: 12
Duplicate date-division combinations: 0
Divisions: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Modeling dataset dimensions: (14748, 14)
Final feature structure validation: PASSED


### 3.14. Save Feature Engineering Outputs

The feature engineering dataset has passed the structural verification checks.

We will save two datasets: the complete feature dataset, which preserves all division-day combinations, and the supervised modeling dataset, which contains only observations with known intervention counts.

Both datasets contain the 12 candidate predictive features developed in this stage. The feature definitions respect the D−3 historical information cutoff.

These outputs will be used in subsequent chronological validation and model training stages.

In [ ]:

# Define output file paths
complete_file = "../../data/interim/modelling/SIM_Features_Complete.csv"
modeling_file = "../../data/interim/modelling/SIM_Features_Modeling.csv"

# Save the complete feature dataset
df_features_complete.to_csv(complete_file, index=False)

# Save the supervised modeling dataset
df_modeling.to_csv(modeling_file, index=False)

# Confirm saved files
print("Complete feature dataset saved:", complete_file)
print("Supervised modeling dataset saved:", modeling_file)

print("\nComplete dataset records:", len(df_features_complete))
print("Modeling dataset records:", len(df_modeling))


Complete feature dataset saved: ../../data/interim/modelling/SIM_Features_Complete.csv
Supervised modeling dataset saved: ../../data/interim/modelling/SIM_Features_Modeling.csv

Complete dataset records: 14754
Modeling dataset records: 14748



### 3.15. Stage Conclusion — Feature Engineering

This stage prepared the candidate predictive features for forecasting daily SIM intervention workloads across six operational divisions.

The dataset contains 12 candidate predictive features:

- **Division:** `DIVISION`
- **Calendar features:** `DayOfWeek`, `Month`, `DayOfMonth`, `IsWeekend`, and `DayOfYear`
- **Lag features:** `Lag_3`, `Lag_7`, `Lag_14`, and `Lag_28`
- **Rolling average features:** `Rolling_Mean_7` and `Rolling_Mean_14`

For each forecast day D, the forecast is prepared on D−1 using historical intervention counts available through D−3. Calendar features describe the forecast date and are known in advance.

The feature engineering process preserved the original daily structure and separated historical calculations by division.

Structural validations confirmed daily continuity within divisions, preservation of the original dataset values, and the expected final feature structure. Selected lag and rolling average calculations were also verified against historical observations.

The feature completeness analysis identified 258 observations with at least one missing predictive feature. These observations were retained for subsequent model-specific preparation.

LightGBM and CatBoost can handle missing numerical features, while Linear Regression will require an appropriate preprocessing strategy.

Two datasets were created:

- `SIM_Features_Complete.csv`: 14,754 division-day observations, including six rows with unknown intervention counts.
- `SIM_Features_Modeling.csv`: 14,748 observations with known intervention counts, retaining the 258 observations with missing predictive features.

The datasets preserve the features needed for subsequent chronological validation and model development.

**Conclusion:** Stage 3 produced a structurally verified feature dataset containing 12 candidate predictors for next-day intervention forecasting.

The predictive usefulness of these features has not yet been established. Subsequent stages will organize the chronological validation, train and evaluate the machine learning models, and compare their performance against the selected baseline.

The reserved final test period remains unevaluated.
